In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg2zn11_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)

# Check space group
ps_bulk = AseAtomsAdaptor().get_structure(alloy)
sga = SpacegroupAnalyzer(ps_bulk, symprec=0.1)

print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")
print(f"Cell: {np.round(alloy.cell.lengths(), 3)} A")
print(f"Angles: {np.round(alloy.cell.angles(), 2)} deg")
print(f"Space group: {sga.get_space_group_symbol()} (#{sga.get_space_group_number()})")
print(f"Composition: {Counter(alloy.get_chemical_symbols())}")

Bulk: 39 atoms, Mg6Zn33
Cell: [8.71 8.71 8.71] A
Angles: [90. 90. 90.] deg
Space group: Pm-3 (#200)
Composition: Counter({'Zn': 33, 'Mg': 6})


In [2]:
slab = surface(alloy, (1, 0, 0), 10, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need {11/2:.4f})")
print(f"Stoichiometric: {'YES' if mg*11 == zn*2 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 390, Mg: 60, Zn: 330
Thickness: 85.7 A
Ratio Zn/Mg: 5.5000 (need 5.5000)
Stoichiometric: YES
Symmetric: False


In [3]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Tolerance: 0.258 A

Total atomic layers: 100

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0     Mg2Zn6     0.000  |       99        Zn1     0.000 NO <---
      1        Zn1     1.383  |       98        Zn7     0.583 NO <---
      2        Zn7     1.966  |       97        Mg1     1.272 NO <---
      3        Mg1     2.656  |       96        Zn2     1.605 NO <---
      4        Zn2     2.988  |       95     Mg2Zn7     2.972 NO <---
      5     Mg2Zn7     4.355  |       94        Zn2     4.339 NO <---
      6        Zn2     5.722  |       93        Mg1     4.671 NO <---
      7        Mg1     6.055  |       92        Zn7     5.361 NO <---
      8        Zn7     6.744  |       91        Zn1     5.944 NO <---
      9        Zn1     7.327  |       90     Mg2Zn6     7.327 NO <---
     10     Mg2Zn6     8.710  |       89        Zn1     8.710 NO <---
     11        Zn1    10.094  |   

In [4]:
print("Searching for symmetric removals...\n")

found_any = False
for bot_rm in range(0, min(20, n//2)):
    for top_rm in range(0, min(20, n//2)):
        if bot_rm == 0 and top_rm == 0:
            continue
        if bot_rm + top_rm > n - 5:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) < 50:
            continue
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                found_any = True
        except:
            continue

if not found_any:
    print("No symmetric sub-slab found")

Searching for symmetric removals...

  bot_rm=0, top_rm=9: 359 atoms, Mg56 Zn303, removed 0+31=31
  bot_rm=0, top_rm=19: 320 atoms, Mg50 Zn270, removed 0+70=70
  bot_rm=1, top_rm=0: 382 atoms, Mg58 Zn324, removed 8+0=8
  bot_rm=1, top_rm=10: 343 atoms, Mg52 Zn291, removed 8+39=47
  bot_rm=2, top_rm=1: 380 atoms, Mg58 Zn322, removed 9+1=10
  bot_rm=2, top_rm=11: 341 atoms, Mg52 Zn289, removed 9+40=49
  bot_rm=3, top_rm=2: 366 atoms, Mg58 Zn308, removed 16+8=24
  bot_rm=3, top_rm=12: 327 atoms, Mg52 Zn275, removed 16+47=63
  bot_rm=4, top_rm=3: 364 atoms, Mg56 Zn308, removed 17+9=26
  bot_rm=4, top_rm=13: 325 atoms, Mg50 Zn275, removed 17+48=65
  bot_rm=5, top_rm=4: 360 atoms, Mg56 Zn304, removed 19+11=30
  bot_rm=5, top_rm=14: 321 atoms, Mg50 Zn271, removed 19+50=69
  bot_rm=6, top_rm=5: 342 atoms, Mg52 Zn290, removed 28+20=48
  bot_rm=6, top_rm=15: 303 atoms, Mg46 Zn257, removed 28+59=87
  bot_rm=7, top_rm=6: 338 atoms, Mg52 Zn286, removed 30+22=52
  bot_rm=7, top_rm=16: 299 atoms, Mg4

In [5]:
# bot_rm=1, top_rm=0: remove 8 atoms from bottom only
removed_bot = layer_idx[0]
keep = []
for j in range(1, n):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_bot)
rem_zn = sum(sym[j] == 'Zn' for j in removed_bot)
print(f"\nRemoved: {len(removed_bot)} atoms (Mg{rem_mg} Zn{rem_zn})")

for j in removed_bot:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab)
keep_set = set(keep)

paired = []
unpaired = []
for j in removed_bot:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    # Check against other removed atoms
    matched = False
    for k in removed_bot:
        if k == j:
            continue
        if np.linalg.norm(slab.positions[k] - inv_cart) < 0.5:
            paired.append((j, k))
            matched = True
            break
    
    if not matched:
        # Check against kept atoms
        dists = np.linalg.norm(slab.get_positions() - inv_cart, axis=1)
        min_d = min(dists[k] for k in keep_set)
        closest = min(keep_set, key=lambda k: dists[k])
        if min_d < 0.5:
            print(f"  idx={j} -> partner in slab (idx={closest}, dist={min_d:.3f})")
        else:
            unpaired.append(j)
            print(f"  idx={j} -> UNPAIRED, inv at ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

print(f"\nPaired within removed: {len(paired)//2}")
print(f"Unpaired: {len(unpaired)}")
unp_mg = sum(sym[j] == 'Mg' for j in unpaired)
unp_zn = sum(sym[j] == 'Zn' for j in unpaired)
print(f"Unpaired: Mg{unp_mg} Zn{unp_zn}")

SG = Pmmm
Inversion: f -> [0.         0.         0.01359833] - f

Removed: 8 atoms (Mg2 Zn6)
  idx=28 Zn (1.383, 4.355, 8.000)
  idx=27 Zn (0.000, 6.728, 8.000)
  idx=21 Zn (6.728, 0.000, 8.000)
  idx=20 Mg (4.355, 2.656, 8.000)
  idx=8 Zn (1.982, 0.000, 8.000)
  idx=9 Zn (0.000, 1.982, 8.000)
  idx=36 Zn (7.327, 4.355, 8.000)
  idx=35 Mg (4.355, 6.055, 8.000)
  idx=28 -> UNPAIRED, inv at (7.327, 4.355, 95.105)
  idx=27 -> UNPAIRED, inv at (8.710, 1.982, 95.105)
  idx=21 -> UNPAIRED, inv at (1.982, 8.710, 95.105)
  idx=20 -> UNPAIRED, inv at (4.355, 6.055, 95.105)
  idx=8 -> UNPAIRED, inv at (6.728, 8.710, 95.105)
  idx=9 -> UNPAIRED, inv at (8.710, 6.728, 95.105)
  idx=36 -> UNPAIRED, inv at (1.383, 4.355, 95.105)
  idx=35 -> UNPAIRED, inv at (4.355, 2.656, 95.105)

Paired within removed: 0
Unpaired: 8
Unpaired: Mg2 Zn6


In [6]:
sc_fixed = slab.copy()
ps_orig = AseAtomsAdaptor().get_structure(slab)

# Mutual pairs: keep first on bottom, move second to inv(first)
move_pairs = [
    (36, 28),   # Zn: move 36 to inv(28)
    (35, 20),   # Mg: move 35 to inv(20)
    (9, 27),    # Zn: move 9 to inv(27)
    (21, 8),    # Zn: move 21 to inv(8)
]

for move_idx, keep_idx in move_pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"Moved {move_idx} ({sym[move_idx]}) -> inv({keep_idx}): "
          f"({old[0]:.3f}, {old[1]:.3f}, {old[2]:.3f}) -> "
          f"({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if mg_f*11 == zn_f*2 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")

Moved 36 (Zn) -> inv(28): (7.327, 4.355, 8.000) -> (7.327, 4.355, 95.105)
Moved 35 (Mg) -> inv(20): (4.355, 6.055, 8.000) -> (4.355, 6.055, 95.105)
Moved 9 (Zn) -> inv(27): (0.000, 1.982, 8.000) -> (8.710, 1.982, 95.105)
Moved 21 (Zn) -> inv(8): (6.728, 0.000, 8.000) -> (6.728, 8.710, 95.105)

Atoms: 390, Mg: 60, Zn: 330
Stoichiometric: YES
Symmetric: True


In [7]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"Atoms: {len(sc_fixed)}, Mg: 60, Zn: 330")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")

view(sc_fixed)

Atoms: 390, Mg: 60, Zn: 330
Thickness: 87.1 A
Area: 75.9 A²


<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [8]:
z_fixed = sc_fixed.get_positions()[:, 2]
print(f"z range: {z_fixed.min():.1f} to {z_fixed.max():.1f}")
print(f"Cell z: {sc_fixed.cell[2][2]:.1f}")

# Find outlier atoms
for i in range(len(sc_fixed)):
    if z_fixed[i] > 90 or z_fixed[i] < 8:
        print(f"  idx={i} {sc_fixed.get_chemical_symbols()[i]} z={z_fixed[i]:.3f}")

z range: 8.0 to 95.1
Cell z: 101.7
  idx=9 Zn z=95.105
  idx=21 Zn z=95.105
  idx=35 Mg z=95.105
  idx=36 Zn z=95.105
  idx=353 Mg z=90.750
  idx=354 Zn z=90.750
  idx=355 Zn z=90.750
  idx=358 Zn z=90.750
  idx=361 Zn z=93.123
  idx=362 Zn z=93.205
  idx=363 Zn z=93.721
  idx=364 Zn z=93.014
  idx=368 Mg z=90.750
  idx=369 Zn z=90.750
  idx=373 Zn z=93.205
  idx=374 Zn z=92.116
  idx=375 Zn z=93.014
  idx=377 Zn z=90.750
  idx=380 Mg z=92.449
  idx=381 Zn z=90.750
  idx=382 Zn z=93.205
  idx=385 Zn z=90.750
  idx=388 Zn z=93.205
  idx=389 Zn z=92.116


In [10]:
pos = sc_fixed.get_positions()
sym_f = np.array(sc_fixed.get_chemical_symbols())

moved_atoms = [9, 21, 35, 36]
for i in moved_atoms:
    dists = np.linalg.norm(pos - pos[i], axis=1)
    dists[i] = 999
    nn_dist = dists.min()
    nn_idx = dists.argmin()
    print(f"idx={i} {sym_f[i]} z={pos[i][2]:.3f} | nearest: idx={nn_idx} {sym_f[nn_idx]} dist={nn_dist:.3f} A z={pos[nn_idx][2]:.3f}")

idx=9 Zn z=95.105 | nearest: idx=373 Zn dist=2.688 A z=93.205
idx=21 Zn z=95.105 | nearest: idx=388 Zn dist=2.688 A z=93.205
idx=35 Mg z=95.105 | nearest: idx=364 Zn dist=3.021 A z=93.014
idx=36 Zn z=95.105 | nearest: idx=375 Zn dist=2.636 A z=93.014


In [11]:
# 10 layers but 2x2 in-plane supercell
slab = surface(alloy, (1, 0, 0), 6, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = sc.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(sc.cell[0], sc.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"6-layer 2x2:")
print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")
print(f"Stoichiometric: {'YES' if mg*11 == zn*2 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

6-layer 2x2:
Atoms: 936, Mg: 144, Zn: 792
Thickness: 50.9 A
Area: 303.5 A²
Stoichiometric: YES
Symmetric: False


In [12]:
z = sc.get_positions()[:, 2]
sym = np.array(sc.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Tolerance: {tol:.3f} A")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
z_min, z_max = z.min(), z.max()
print(f"Total layers: {n}")

# Search for symmetric removal
print("\nSearching...\n")

found_any = False
for bot_rm in range(0, min(15, n//2)):
    for top_rm in range(0, min(15, n//2)):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) < 100:
            continue
        tr = sc[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                found_any = True
        except:
            continue

if not found_any:
    print("No symmetric sub-slab found")

Tolerance: 0.258 A
Total layers: 60

Searching...

  bot_rm=0, top_rm=9: 812 atoms, Mg128 Zn684, removed 0+124=124
  bot_rm=1, top_rm=0: 904 atoms, Mg136 Zn768, removed 32+0=32
  bot_rm=1, top_rm=10: 748 atoms, Mg112 Zn636, removed 32+156=188
  bot_rm=2, top_rm=1: 896 atoms, Mg136 Zn760, removed 36+4=40
  bot_rm=2, top_rm=11: 740 atoms, Mg112 Zn628, removed 36+160=196
  bot_rm=3, top_rm=2: 840 atoms, Mg136 Zn704, removed 64+32=96
  bot_rm=3, top_rm=12: 684 atoms, Mg112 Zn572, removed 64+188=252
  bot_rm=4, top_rm=3: 832 atoms, Mg128 Zn704, removed 68+36=104
  bot_rm=4, top_rm=13: 676 atoms, Mg104 Zn572, removed 68+192=260
  bot_rm=5, top_rm=4: 816 atoms, Mg128 Zn688, removed 76+44=120
  bot_rm=5, top_rm=14: 660 atoms, Mg104 Zn556, removed 76+200=276
  bot_rm=6, top_rm=5: 744 atoms, Mg112 Zn632, removed 112+80=192
  bot_rm=7, top_rm=6: 728 atoms, Mg112 Zn616, removed 120+88=208
  bot_rm=8, top_rm=7: 720 atoms, Mg104 Zn616, removed 124+92=216
  bot_rm=9, top_rm=8: 664 atoms, Mg104 Zn560,

In [13]:
removed_bot = layer_idx[0]
keep = []
for j in range(1, n):
    keep.extend(layer_idx[j])

trimmed = sc[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_bot)
rem_zn = sum(sym[j] == 'Zn' for j in removed_bot)
print(f"\nRemoved: {len(removed_bot)} atoms (Mg{rem_mg} Zn{rem_zn})")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(sc)
keep_set = set(keep)

unpaired = []
paired_in_removed = []
for j in removed_bot:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    # Check against other removed atoms
    matched = False
    for k in removed_bot:
        if k == j:
            continue
        if np.linalg.norm(sc.positions[k] - inv_cart) < 0.5:
            paired_in_removed.append((j, k))
            matched = True
            break
    
    if not matched:
        dists = np.linalg.norm(sc.get_positions() - inv_cart, axis=1)
        min_d = min(dists[k] for k in keep_set)
        if min_d < 0.5:
            closest = min(keep_set, key=lambda k: dists[k])
            print(f"  idx={j} {sym[j]} -> partner in slab (idx={closest})")
        else:
            unpaired.append(j)

print(f"\nMutual pairs within removed: {len(paired_in_removed)//2}")
print(f"Unpaired: {len(unpaired)}")

# Find mutual pairs
used = set()
mutual = []
for j1, j2 in paired_in_removed:
    if j1 not in used and j2 not in used:
        mutual.append((j1, j2))
        used.add(j1)
        used.add(j2)

print(f"Distinct mutual pairs: {len(mutual)}")

# Reconstruct
sc_fixed = sc.copy()
for keep_idx, move_idx in mutual:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    old = sc_fixed.positions[move_idx].copy()
    sc_fixed.positions[move_idx] = inv_cart
    print(f"  Moved {move_idx} -> inv({keep_idx})")

# Handle any remaining unpaired with brute force if needed
if len(unpaired) > 0 and len(unpaired) % 2 == 0:
    print(f"\n{len(unpaired)} unpaired atoms — trying brute force...")
    unp_mg = [j for j in unpaired if sym[j] == 'Mg']
    unp_zn = [j for j in unpaired if sym[j] == 'Zn']
    
    found = False
    for keep_mg in combinations(unp_mg, len(unp_mg)//2) if unp_mg else [()]:
        move_mg = [j for j in unp_mg if j not in keep_mg]
        for keep_zn in combinations(unp_zn, len(unp_zn)//2) if unp_zn else [()]:
            move_zn = [j for j in unp_zn if j not in keep_zn]
            for perm_mg in permutations(keep_mg) if keep_mg else [()]:
                for perm_zn in permutations(keep_zn) if keep_zn else [()]:
                    sc_test = sc_fixed.copy()
                    for m, k in zip(move_mg, perm_mg):
                        frac = ps_orig[k].frac_coords
                        inv_frac = (inv_translation - frac) % 1.0
                        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                        sc_test.positions[m] = inv_cart
                    for m, k in zip(move_zn, perm_zn):
                        frac = ps_orig[k].frac_coords
                        inv_frac = (inv_translation - frac) % 1.0
                        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                        sc_test.positions[m] = inv_cart
                    
                    try:
                        ps_test = AseAtomsAdaptor().get_structure(sc_test)
                        slab_test = Slab(ps_test.lattice, ps_test.species, ps_test.frac_coords,
                            miller_index=(1,0,0), oriented_unit_cell=ps_test, shift=0,
                            scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                        if slab_test.is_symmetric():
                            print("  FOUND!")
                            sc_fixed = sc_test
                            found = True
                            break
                    except:
                        continue
                if found:
                    break
            if found:
                break
        if found:
            break

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if mg_f*11 == zn_f*2 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")

SG = Pmmm
Inversion: f -> [0.         0.         0.02068252] - f

Removed: 32 atoms (Mg8 Zn24)

Mutual pairs within removed: 0
Unpaired: 32
Distinct mutual pairs: 0

32 unpaired atoms — trying brute force...
  FOUND!

Atoms: 936, Mg: 144, Zn: 792
Stoichiometric: YES
Symmetric: True
Thickness: 52.3 A
Area: 303.5 A²


In [14]:
z_fixed = sc_fixed.get_positions()[:, 2]
print(f"z range: {z_fixed.min():.1f} to {z_fixed.max():.1f}")
print(f"Cell z: {sc_fixed.cell[2][2]:.1f}")

# Check nearest neighbours of top-most atoms
top_atoms = np.where(z_fixed > z_fixed.max() - 2.0)[0]
sym_f = np.array(sc_fixed.get_chemical_symbols())
pos = sc_fixed.get_positions()

print(f"\nTop surface atoms (within 2A of max z):")
for i in top_atoms:
    dists = np.linalg.norm(pos - pos[i], axis=1)
    dists[i] = 999
    nn_dist = dists.min()
    nn_idx = dists.argmin()
    print(f"  idx={i} {sym_f[i]} z={z_fixed[i]:.3f} | nn: idx={nn_idx} dist={nn_dist:.3f}")

view(sc_fixed)

z range: 8.0 to 60.3
Cell z: 66.9

Top surface atoms (within 2A of max z):
  idx=8 Zn z=60.263 | nn: idx=451 dist=2.688
  idx=9 Zn z=60.263 | nn: idx=460 dist=2.688
  idx=35 Mg z=60.263 | nn: idx=910 dist=3.021
  idx=205 Zn z=58.281 | nn: idx=206 dist=2.688
  idx=206 Zn z=58.363 | nn: idx=205 dist=2.688
  idx=207 Zn z=58.879 | nn: idx=218 dist=2.636
  idx=217 Zn z=58.363 | nn: idx=476 dist=2.688
  idx=226 Zn z=58.363 | nn: idx=439 dist=2.688
  idx=232 Zn z=58.363 | nn: idx=907 dist=2.688
  idx=242 Zn z=60.263 | nn: idx=919 dist=2.688
  idx=243 Zn z=60.263 | nn: idx=921 dist=2.636
  idx=261 Zn z=60.263 | nn: idx=910 dist=2.636
  idx=262 Zn z=60.263 | nn: idx=208 dist=2.636
  idx=270 Zn z=60.263 | nn: idx=219 dist=2.636
  idx=439 Zn z=58.281 | nn: idx=226 dist=2.688
  idx=440 Zn z=58.363 | nn: idx=439 dist=2.688
  idx=441 Zn z=58.879 | nn: idx=233 dist=2.636
  idx=451 Zn z=58.363 | nn: idx=8 dist=2.688
  idx=460 Zn z=58.363 | nn: idx=9 dist=2.688
  idx=466 Zn z=58.363 | nn: idx=463 dist=

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [15]:
z_fixed = sc_fixed.get_positions()[:, 2]
z_bulk_top = np.sort(z_fixed)[-50]  # approximate top of main slab body

print(f"z range: {z_fixed.min():.1f} to {z_fixed.max():.1f}")
print(f"Main slab body top (approx): {z_bulk_top:.1f}")

# Find atoms that are above the main body
outliers = np.where(z_fixed > z_bulk_top + 2.0)[0]
print(f"\nOutlier atoms above main slab body:")
sym_f = np.array(sc_fixed.get_chemical_symbols())
for i in outliers:
    print(f"  idx={i} {sym_f[i]} z={z_fixed[i]:.3f}")

z range: 8.0 to 60.3
Main slab body top (approx): 57.6

Outlier atoms above main slab body:
  idx=8 Zn z=60.263
  idx=9 Zn z=60.263
  idx=35 Mg z=60.263
  idx=242 Zn z=60.263
  idx=243 Zn z=60.263
  idx=261 Zn z=60.263
  idx=262 Zn z=60.263
  idx=270 Zn z=60.263
  idx=476 Zn z=60.263
  idx=477 Zn z=60.263
  idx=488 Mg z=60.263
  idx=503 Mg z=60.263
  idx=710 Zn z=60.263
  idx=711 Zn z=60.263
  idx=722 Mg z=60.263
  idx=723 Zn z=60.263


In [16]:
z_bulk_bot = np.sort(z_fixed)[50]  # approximate bottom of main slab body
print(f"Main slab body bottom (approx): {z_bulk_bot:.1f}")

outliers_bot = np.where(z_fixed < z_bulk_bot - 2.0)[0]
print(f"\nOutlier atoms below main slab body:")
for i in outliers_bot:
    print(f"  idx={i} {sym_f[i]} z={z_fixed[i]:.3f}")

# Also check: where are the 16 lowest-z atoms?
lowest = np.argsort(z_fixed)[:20]
print(f"\n20 lowest z atoms:")
for i in lowest:
    print(f"  idx={i} {sym_f[i]} z={z_fixed[i]:.3f}")

Main slab body bottom (approx): 10.7

Outlier atoms below main slab body:
  idx=20 Mg z=8.000
  idx=21 Zn z=8.000
  idx=27 Zn z=8.000
  idx=28 Zn z=8.000
  idx=36 Zn z=8.000
  idx=254 Mg z=8.000
  idx=255 Zn z=8.000
  idx=269 Mg z=8.000
  idx=489 Zn z=8.000
  idx=495 Zn z=8.000
  idx=496 Zn z=8.000
  idx=504 Zn z=8.000
  idx=729 Zn z=8.000
  idx=730 Zn z=8.000
  idx=737 Mg z=8.000
  idx=738 Zn z=8.000

20 lowest z atoms:
  idx=729 Zn z=8.000
  idx=730 Zn z=8.000
  idx=255 Zn z=8.000
  idx=504 Zn z=8.000
  idx=21 Zn z=8.000
  idx=489 Zn z=8.000
  idx=20 Mg z=8.000
  idx=254 Mg z=8.000
  idx=495 Zn z=8.000
  idx=36 Zn z=8.000
  idx=496 Zn z=8.000
  idx=738 Zn z=8.000
  idx=28 Zn z=8.000
  idx=269 Mg z=8.000
  idx=737 Mg z=8.000
  idx=27 Zn z=8.000
  idx=248 Zn z=9.383
  idx=716 Zn z=9.383
  idx=482 Zn z=9.383
  idx=14 Zn z=9.383


In [17]:
ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg2zn11_100_6layers_reconstructed_ase.data")

thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"Saved: slabs/mg2zn11_100_6layers_2x2_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg2zn11_100_6layers_2x2_reconstructed_ase.data
  Atoms: 936
  Thickness: 52.3 A
  Area: 303.5 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [4]:
#unrelaxed surface energy calculation
E_slab = -1028.075        # eV
E_bulk_per_fu =  -44.119792 / 3  # eV per formula unit 
n = 936 / 13                 # formula units in slab = 32
A = 303.5             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.706597 eV
n * E_bulk = -1058.87501 eV
E_slab - n*E_bulk = 30.80001 eV
S = 0.050741 eV/Ang^2
S = 0.8130 J/m^2


In [5]:
#relaxed surface energy calculation
E_slab_relaxed = -1035.11756673497    # eV
E_bulk_per_fu = -44.119792 / 3  # eV per formula unit
n = 936 / 13                      # 32 formula units
A = 303.5                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 =  0.8130
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 23.75744 eV
S relaxed = 0.039139 eV/Ang^2
S relaxed = 0.6271 J/m^2

Unrelaxed S = 0.8130 J/m^2
Relaxed S   = 0.6271 J/m^2
Relaxation energy = 0.1859 J/m^2
